# Import

In [ ]:
from scipy.io import loadmat
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
from mapd import Trial, Table, Sinq
import h5py
import glob
import numpy as np
import pickle
import plotly.express as px
import matplotlib.colors as mcolors
# import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
import matplotlib as mpl
mpl.rcParams.update(mpl.rcParamsDefault)  # reset to defaults
mpl.rcParams['pdf.fonttype'] = 42         # embed fonts as text, not paths
mpl.rcParams['svg.fonttype'] = 'none'     # keep text editable in SVG
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['font.size'] = 11

import seaborn as sns

import random

%load_ext autoreload
%autoreload 2

import mapd.sentinels  # load once
%aimport -mapd.sentinels

%matplotlib widget

def refresh_table_addons():
    import importlib
    import mapd.table_plotters as tps
    import mapd.table_scalars as tbs
    importlib.reload(tps)
    importlib.reload(tbs)

    for name in dir(tps):
        if name.startswith('_'):
            continue
        attr = getattr(tps, name)
        setattr(Table, name[len("plot_"):], attr)
    
    for name in dir(tbs):
        if name.startswith('_'):
            continue
        if name.startswith('compute_'):
            attr = getattr(tbs, name)
            setattr(Table, name[len("compute_"):], attr)

# Load Figure 3 sinq with norpA flies

In [ ]:
sinq = Sinq(sinqname='Figure3')
print(sinq.__repr__())
# sinq.df

# sinq1 = Sinq(sinqname='Figure1')
# sinq.df.loc[sinq1.df.index] = sinq1.df
# sinq.save()
# sinq.df

# Look at how long it takes for norpa flies to move

In [ ]:
print([i for i in sinq.df['genotype'].unique()])
op_df = sinq.df.copy()   # Don't mess with sinq

geno_map = {'Hot-Cell-Gal4 (test)': 'HC',
 'Hot-Cell-LexA_Chr;81A06_pJFRC7':  'HC',
 'Hot-Cell-LexA_Chr;35c09_pJFRC7':  'HC',
 'Hot-Cell-LexA_Chr;35C09_pJFRC7':  'HC',
 'Hot-Cell-LexA_Chr;78E05_pJFRC7':  'HC',
 'Hot-Cell-LexA_Chr;31H05_pJFRC7':  'HC',
 'iav_Kir2_1':                      'w;iav>kir2.1',
 '31H05_pJFRC7':                    'w',
 'SS61350_pJFRC7':                  '+',
 'iav-Gal4_+;+;UAS-Kir2_1':         '+;iav>kir2.1',
 '+;iav-Gal4_UAS-Kir2_1':           '+;iav>kir2.1',
 '+;31H05-Gal4_pJFRC7':             '+',
 '+;TH-Gal4_UAS-Kir2_1':            '+;TH>kir2.1',
 '+;TH-Gal4_pJFRC7':                '+',
 'w, NorpA':                        'NorpA',
 'norpAE55':                      'NorpA',
 '+;pJFRC7;31H05-Gal4':             '+',
 '+;pJFRC7;SS61350':              '+',
 }

op_df['geno_smpl'] = sinq.df['genotype'].map(geno_map)


learner_types = ['+', 'HC','+;TH-Gal4_pJFRC7']
non_learner_types = ['w','w;iav>kir2.1','w,NorpA','+;iav>kir2.1']
maybe_learner_types = ['+;TH>kir2.1',]
op_df['learn_geno'] = op_df['geno_smpl'].apply(
    lambda x: any(t == x for t in learner_types)
)
op_df['is_learn'] = op_df['learn_geno']
op_df['learn_note'] = ''
op_df['learn_class'] = op_df['is_learn'].astype(int)

In [ ]:
all(op_df['is_learn'] == op_df['learn_geno'])

In [ ]:
op_df.loc[op_df['geno_smpl'] == 'NorpA']

# Fold blocks and average outcomes across blocks

In [ ]:
op_df.loc[op_df['geno_smpl']=='w']

In [ ]:
from collections import defaultdict

for g in ['w','+;iav>kir2.1','w;iav>kir2.1','+;TH>kir2.1']:

    norm = True
    total_outcomes = None
    total_success = None
    total_probe = None
    outcome_dict = defaultdict(dict)

    g_flies =    op_df.loc[op_df['geno_smpl']==g]
    for dfc in reversed(g_flies.index):
        export_path = f'./Figure3/folded_blocks/{dfc}/exports'
        pklpat = f'{export_path}/folded_outcome_counts_{dfc}.pkl'
        file_path = glob.glob(pklpat)
        if file_path:
            file_path = file_path[0]
            print(file_path)

        outcome_dict[dfc] = pd.read_pickle(file_path)

        if total_outcomes is None:
            total_outcomes = outcome_dict[dfc].copy()
        else:
            total_outcomes = total_outcomes.add(outcome_dict[dfc], fill_value=0)

    plt.close('all')
    fig = Figure(figsize=(8,8))
    lines = defaultdict(dict)

    colors = {'hi': '#1f77b4', 
            'lo': '#d62728'}
    colors = {'as_off': "#169400", 
                'no_as': "#000cad",
                'no_as_mv': "#6068cb",
                'success': "#009480",
                'to': "#FF0000",
                'probe_as': "#FF30FC",
            }

    hi_x = total_outcomes.xs('hi', level='pyasState').index
    x = {'hi': hi_x,
        'lo': total_outcomes.xs('lo', level='pyasState').index + hi_x[-1] + 1}

    outcome_to_ax = {'as_off':1,
                    'no_as':2,
                    'no_as_mv':2,
                    'success':3,
                    'to':3,
                    'probe_as':3,}

    ax_ids = sorted(set(outcome_to_ax.values()))
    ax_id_to_ax = {}
    for i, ax_id in enumerate(ax_ids, start=1):
        ax_id_to_ax[ax_id] = fig.add_subplot(len(ax_ids), 1, i)

    # for dfc in reversed(sinq2.df.index):
    for dfc in outcome_dict.keys():
        for outcome in outcome_to_ax.keys():
            for st in ['hi','lo']:
                outcomes = outcome_dict[dfc]
                ax = ax_id_to_ax[outcome_to_ax[outcome]]
                hi_y = outcomes.xs(st, level='pyasState').get(outcome)
                if norm:
                    hi_y = hi_y / outcomes.xs(st, level='pyasState').get('max_cnts')

                # ax.scatter(
                #             x[st], hi_y, marker='.',s=5, alpha=0.9, c=colors[outcome]
                #         )
                if norm:
                    ax.set_ylim([0, 1])
                else:
                    ax.set_ylim([0, np.max(outcomes.xs(st, level='pyasState').get('max_cnts'))])


    for outcome in outcome_to_ax.keys():
        for st in ['hi','lo']:
            ax = ax_id_to_ax[outcome_to_ax[outcome]]
            hi_y = total_outcomes.xs(st, level='pyasState').get(outcome)
            if norm:
                hi_y = hi_y / total_outcomes.xs(st, level='pyasState').get('max_cnts')

            lines[outcome][st], = ax.plot(
                        x[st], hi_y, label=f'{outcome} ({st})', linewidth=1, alpha=0.9, color=colors[outcome]
                    )
            if norm:
                ax.set_ylim([0, 1])
            else:
                ax.set_ylim([0, np.max(total_outcomes.xs(st, level='pyasState').get('max_cnts'))])

    ax_id_to_ax[1].set_ylabel('as_off')
    ax_id_to_ax[2].set_ylabel('no_as')
    ax_id_to_ax[3].set_ylabel('success')
    ax_id_to_ax[3].legend()
    fig.savefig(f'./Figure3/outcomes_over_blocks_{_get_gstr(g)}.svg',format='svg')
    display(fig)

In [ ]:
pd.Index([i for i in range(500)])

In [ ]:
# index=
T.df.loc[T.df.index[50:450]]

In [ ]:
sinq.drop_tables()

In [ ]:
for g in ['w','+;iav>kir2.1','+;TH>kir2.1']:
    g_flies =    op_df.loc[op_df['geno_smpl']==g]
    for dfc in reversed(g_flies.index):
        T = sinq.restore_table(dfc)
        T.extract_trial_properties()
        T.fig_folder = f'./Figure3/heatmaps/{_get_gstr(g)}'
        idx=T.df.loc[T.df.index[50:450]].index
        print(f'{dfc}: making heat map for {len(idx)} trials')
        try:
            fig,ax,df = T.plot_probe_position_heatmap(index=idx,cmin=-500,cmax=10,format='svg')
        except Exception as e:
            print(f'{e}')